# Create Network Evolution GIFs/Videos

This notebook creates animated GIFs showing the temporal evolution of migration networks in China,
including both geographic map views and hyperbolic space representations.

In [17]:
import os
import re
from pathlib import Path
from collections import defaultdict
import numpy as np
from PIL import Image, ImageDraw, ImageFont
import imageio
import matplotlib.pyplot as plt
from typing import List, Dict, Tuple

In [18]:
# Configuration
OUTPUT_DIR = Path('hyperbolic_outputs_by_characteristic')
GIF_OUTPUT_DIR = Path('network_evolution_gifs')
GIF_OUTPUT_DIR.mkdir(exist_ok=True)

# GIF settings
DURATION = 500  # milliseconds per frame
LOOP = 0  # 0 means loop forever

In [19]:
def parse_filename(filename: str) -> Dict:
    """
    Parse filename to extract granularity, year, and visualization type.
    
    Example: economical_county_1985_flows_hyperbolic_communities.png
    Returns: {'granularity': 'county', 'year': 1985, 'viz_type': 'hyperbolic_communities'}
    """
    # Pattern: economical_{granularity}_{year}_flows_{viz_type}.png
    pattern = r'economical_(\w+)_(\d+)_flows_(\w+(?:_\w+)*)\.png'
    match = re.match(pattern, filename)
    
    if match:
        return {
            'granularity': match.group(1),
            'year': int(match.group(2)),
            'viz_type': match.group(3)
        }
    return None

def get_all_images(output_dir: Path) -> Dict:
    """
    Scan directory and organize images by granularity and visualization type.
    """
    images = defaultdict(list)
    
    for file in output_dir.glob('*.png'):
        parsed = parse_filename(file.name)
        if parsed and parsed['viz_type'] != 'full':  # Skip 'full' (all years combined)
            key = (parsed['granularity'], parsed['viz_type'])
            images[key].append({
                'path': file,
                'year': parsed['year']
            })
    
    # Sort by year
    for key in images:
        images[key] = sorted(images[key], key=lambda x: x['year'])
    
    return images

In [20]:
def add_year_label(img: Image.Image, year: int, position: str = 'top-left') -> Image.Image:
    """
    Add year label to image.
    """
    img_copy = img.copy()
    draw = ImageDraw.Draw(img_copy)
    
    # Try to use a nice font, fall back to default if not available
    try:
        font = ImageFont.truetype("arial.ttf", 60)
    except:
        font = ImageFont.load_default()
    
    text = f"Year: {year}"
    
    # Get text bounding box
    bbox = draw.textbbox((0, 0), text, font=font)
    text_width = bbox[2] - bbox[0]
    text_height = bbox[3] - bbox[1]
    
    # Position
    margin = 20
    if position == 'top-left':
        x, y = margin, margin
    elif position == 'top-right':
        x, y = img_copy.width - text_width - margin, margin
    elif position == 'bottom-left':
        x, y = margin, img_copy.height - text_height - margin
    else:  # bottom-right
        x, y = img_copy.width - text_width - margin, img_copy.height - text_height - margin
    
    # Draw background rectangle
    padding = 10
    draw.rectangle(
        [x - padding, y - padding, x + text_width + padding, y + text_height + padding],
        fill='white',
        outline='black',
        width=2
    )
    
    # Draw text
    draw.text((x, y), text, fill='black', font=font)
    
    return img_copy

In [21]:
def create_gif(image_list: List[Dict], output_path: Path, duration: int = 500, add_labels: bool = True):
    """
    Create GIF from list of images.
    """
    frames = []
    
    for img_info in image_list:
        img = Image.open(img_info['path'])
        
        if add_labels:
            img = add_year_label(img, img_info['year'])
        
        frames.append(np.array(img))
    
    # Save as GIF
    imageio.mimsave(output_path, frames, duration=duration, loop=0)
    print(f"Created GIF: {output_path}")
    print(f"  - Number of frames: {len(frames)}")
    print(f"  - Years: {image_list[0]['year']} to {image_list[-1]['year']}")

In [22]:
def create_side_by_side_gif(map_images: List[Dict], hyperbolic_images: List[Dict], 
                           output_path: Path, duration: int = 500):
    """
    Create side-by-side comparison GIF (map + hyperbolic).
    """
    frames = []
    
    for map_info, hyp_info in zip(map_images, hyperbolic_images):
        # Ensure they're from the same year
        assert map_info['year'] == hyp_info['year'], "Year mismatch!"
        
        # Load images
        map_img = Image.open(map_info['path'])
        hyp_img = Image.open(hyp_info['path'])
        
        # Resize to same height if needed
        target_height = min(map_img.height, hyp_img.height)
        
        if map_img.height != target_height:
            aspect = map_img.width / map_img.height
            map_img = map_img.resize((int(target_height * aspect), target_height), Image.LANCZOS)
        
        if hyp_img.height != target_height:
            aspect = hyp_img.width / hyp_img.height
            hyp_img = hyp_img.resize((int(target_height * aspect), target_height), Image.LANCZOS)
        
        # Create combined image
        total_width = map_img.width + hyp_img.width + 20  # 20px gap
        combined = Image.new('RGB', (total_width, target_height), 'white')
        combined.paste(map_img, (0, 0))
        combined.paste(hyp_img, (map_img.width + 20, 0))
        
        # Add year label
        combined = add_year_label(combined, map_info['year'], position='top-right')
        
        frames.append(np.array(combined))
    
    # Save as GIF
    imageio.mimsave(output_path, frames, duration=duration, loop=0)
    print(f"Created side-by-side GIF: {output_path}")
    print(f"  - Number of frames: {len(frames)}")
    print(f"  - Years: {map_images[0]['year']} to {map_images[-1]['year']}")

In [23]:
# Get all images organized by granularity and type
print("Scanning images...")
images = get_all_images(OUTPUT_DIR)

print(f"\nFound images for:")
for key, img_list in images.items():
    granularity, viz_type = key
    print(f"  - {granularity} / {viz_type}: {len(img_list)} images")

Scanning images...

Found images for:
  - county / hyperbolic_communities: 37 images
  - county / hyperbolic_flow: 37 images
  - county / map_flow_strength: 37 images
  - prefecture / hyperbolic_communities: 37 images
  - prefecture / hyperbolic_flow: 37 images
  - prefecture / map_flow_strength: 37 images
  - province / hyperbolic_communities: 37 images
  - province / hyperbolic_flow: 37 images
  - province / map_flow_strength: 37 images


## Create Individual GIFs

Create separate GIFs for each granularity and visualization type.

In [ ]:
print("Creating individual GIFs...\n")

for key, img_list in images.items():
    granularity, viz_type = key
    output_path = GIF_OUTPUT_DIR / f"{granularity}_{viz_type}_evolution.gif"
    create_gif(img_list, output_path, duration=DURATION)

Creating individual GIFs...



## Create Side-by-Side Comparison GIFs

Create GIFs showing map and hyperbolic views side by side.

In [ ]:
print("\nCreating side-by-side comparison GIFs...\n")

granularities = ['county', 'prefecture', 'province']
hyperbolic_types = ['hyperbolic_communities', 'hyperbolic_flow']

for granularity in granularities:
    map_key = (granularity, 'map_flow_strength')
    
    if map_key not in images:
        continue
    
    for hyp_type in hyperbolic_types:
        hyp_key = (granularity, hyp_type)
        
        if hyp_key not in images:
            continue
        
        # Get common years
        map_years = {img['year'] for img in images[map_key]}
        hyp_years = {img['year'] for img in images[hyp_key]}
        common_years = sorted(map_years & hyp_years)
        
        if not common_years:
            continue
        
        # Filter to common years
        map_imgs = [img for img in images[map_key] if img['year'] in common_years]
        hyp_imgs = [img for img in images[hyp_key] if img['year'] in common_years]
        
        # Sort by year
        map_imgs = sorted(map_imgs, key=lambda x: x['year'])
        hyp_imgs = sorted(hyp_imgs, key=lambda x: x['year'])
        
        output_path = GIF_OUTPUT_DIR / f"{granularity}_map_vs_{hyp_type}.gif"
        create_side_by_side_gif(map_imgs, hyp_imgs, output_path, duration=DURATION)

## Create MP4 Videos (Higher Quality)

Optionally create MP4 videos for better quality and smaller file sizes.

In [ ]:
def create_mp4(image_list: List[Dict], output_path: Path, fps: int = 2, add_labels: bool = True):
    """
    Create MP4 video from list of images.
    """
    frames = []
    
    for img_info in image_list:
        img = Image.open(img_info['path'])
        
        if add_labels:
            img = add_year_label(img, img_info['year'])
        
        frames.append(np.array(img))
    
    # Save as MP4
    imageio.mimsave(output_path, frames, fps=fps, codec='libx264', pixelformat='yuv420p')
    print(f"Created MP4: {output_path}")
    print(f"  - Number of frames: {len(frames)}")
    print(f"  - Years: {image_list[0]['year']} to {image_list[-1]['year']}")

In [ ]:
# Create MP4 videos for selected visualizations
print("\nCreating MP4 videos...\n")

VIDEO_FPS = 2  # 2 frames per second (slower = easier to see details)

for key, img_list in images.items():
    granularity, viz_type = key
    output_path = GIF_OUTPUT_DIR / f"{granularity}_{viz_type}_evolution.mp4"
    create_mp4(img_list, output_path, fps=VIDEO_FPS)

## Summary

All GIFs and videos have been created in the `network_evolution_gifs` directory.

**Individual visualizations:**
- `{granularity}_{viz_type}_evolution.gif/mp4` - Individual evolution animations

**Side-by-side comparisons:**
- `{granularity}_map_vs_{hyperbolic_type}.gif` - Map and hyperbolic views together

You can adjust the `DURATION` parameter (milliseconds per frame) for GIFs or `VIDEO_FPS` (frames per second) for videos to control playback speed.

In [ ]:
# List all created files
print("\n" + "="*60)
print("Created files:")
print("="*60)

for file in sorted(GIF_OUTPUT_DIR.glob('*')):
    size_mb = file.stat().st_size / (1024 * 1024)
    print(f"{file.name:60s} {size_mb:>8.2f} MB")